# PlagIA Enterprise · 02 — Preprocesamiento Radiométrico (OpenCV CLAHE) y Tiling Geoespacial (Rasterio)

**Subsistema de Visión Computacional y Geoprocesamiento (`ml_engine`)**

### Objetivos Técnicos
1. Verificar la corrección radiométrica **CLAHE en espacio LAB (OpenCV)** sobre capturas de arroz.
2. Construir y validar los `DataLoader` de **PyTorch** con aumento de datos **Albumentations** y `pin_memory` para CUDA.
3. Generar y seccionar un ortomosaico **GeoTIFF (`EPSG:4326`)** con **Rasterio** sobre el distrito de riego de El Espinal (Tolima), conservando la transformación afín GPS de cada parche.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from ml_engine.config import ARTIFACTS_DIR, BATCH_SIZE, SPLIT_MANIFEST_PATH
from ml_engine.preprocessing import (
    build_dataloaders,
    denormalize_tensor,
    load_and_prepare_image,
)
from ml_engine.geospatial import (
    create_sample_orthomosaic_geotiff,
    slice_orthomosaic_into_patches,
)

## 1. Corrección Radiométrica CLAHE + DataLoaders PyTorch

In [ ]:
split_df = pd.read_csv(SPLIT_MANIFEST_PATH)
loaders = build_dataloaders(split_df, batch_size=BATCH_SIZE, apply_clahe=True)

x_batch, y_batch = next(iter(loaders["train"]))
print("Shape del batch NCHW:", x_batch.shape, "| dtype:", x_batch.dtype)
print("Etiquetas del lote  :", y_batch[:8].tolist())

sample_path = split_df.iloc[0]["path"]
raw_rgb = load_and_prepare_image(sample_path, apply_clahe=False)
clahe_rgb = load_and_prepare_image(sample_path, apply_clahe=True)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(raw_rgb)
axes[0].set_title("Captura Original")
axes[0].axis("off")

axes[1].imshow(clahe_rgb)
axes[1].set_title("Corrección Radiométrica CLAHE")
axes[1].axis("off")

axes[2].imshow(denormalize_tensor(x_batch[0]))
axes[2].set_title("Tensor Albumentations (Train)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 2. Particionado de Ortomosaicos GeoTIFF con Rasterio

In [ ]:
sample_imgs = [Path(p) for p in split_df["path"].head(9).tolist()]
sample_tif = create_sample_orthomosaic_geotiff(
    ARTIFACTS_DIR / "orthomosaics" / "lote_espinal_sample.tif",
    zone_key="EL_ESPINAL",
    source_images=sample_imgs,
)
patches = slice_orthomosaic_into_patches(sample_tif, patch_size=224)
print(f"Ortomosaico generado en: {sample_tif}")
print(f"Total parches georreferenciados extraídos: {len(patches)}")
print("Primer parche GPS:", {k: v for k, v in patches[0].items() if k != "image_rgb"})